In [2]:
import pandas as pd

# Reading in the Data

In [3]:
df = pd.read_csv("../datasets/raw/lbnl_solar_data.csv")
df.head()

,PPA Execution Date,Capacity (MW),Term (years),CAISO,West (non-ISO),MISO,SPP,ERCOT,PJM,NYISO,ISO-NE,Southeast (non-ISO),Hawaii
0,9/1/06,7.0,20,NaN,280.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,6/25/07,5.0,20,253.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7/1/08,550.0,25,170.77,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,7/23/08,210.0,25,148.73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12/19/08,10.0,20,196.15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The columns are as follows:
- Date: in the formate MM/DD/YY, and not stored as a datetime type yet
- Capacity in Megawatts,
- Term in Years
- The rest of the columns are the regions in the dataset this csv comes from.

In [4]:
df.dtypes

PPA Execution Date      object
Capacity (MW)          float64
Term (years)             int64
CAISO                  float64
West (non-ISO)         float64
MISO                   float64
SPP                    float64
ERCOT                  float64
PJM                    float64
NYISO                  float64
ISO-NE                 float64
Southeast (non-ISO)    float64
Hawaii                 float64
dtype: object

# Cleaning and Saving Data

We need to group by year, so we create a new "Year" column.

In [5]:
df["Year"] = pd.to_datetime(df["PPA Execution Date"], format="%m/%d/%y").dt.year
df.head()

,PPA Execution Date,Capacity (MW),Term (years),CAISO,West (non-ISO),MISO,SPP,ERCOT,PJM,NYISO,ISO-NE,Southeast (non-ISO),Hawaii,Year
0,9/1/06,7.0,20,NaN,280.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006
1,6/25/07,5.0,20,253.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007
2,7/1/08,550.0,25,170.77,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008
3,7/23/08,210.0,25,148.73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008
4,12/19/08,10.0,20,196.15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008


Here we reshape the dataframe by adding a "Region" column and a "Price ($/MWh)" column.
This changes the current format which has a column for each region where the values are the prices.

In [6]:
region_cols = ["CAISO", "West (non-ISO)", "MISO", "SPP", "ERCOT", "PJM", "NYISO", "ISO-NE", "Southeast (non-ISO)", "Hawaii"]
df = df.melt(id_vars=["PPA Execution Date", "Capacity (MW)", "Term (years)", "Year"], value_vars=region_cols, var_name="Region", value_name="Price ($/MWh)")
df = df.dropna(subset=["Price ($/MWh)"])
df.head()

,PPA Execution Date,Capacity (MW),Term (years),Year,Region,Price ($/MWh)
1,6/25/07,5.0,20,2007,CAISO,253.19
2,7/1/08,550.0,25,2008,CAISO,170.77
3,7/23/08,210.0,25,2008,CAISO,148.73
4,12/19/08,10.0,20,2008,CAISO,196.15
9,5/8/09,230.0,25,2009,CAISO,165.87


Now, to get the average price per year per region, which we will be using later on, we need to group by region and year and take the mean of the prices.

We also are only interested in the years 2015 and later

In [7]:
df = df.groupby(["Region", "Year"])["Price ($/MWh)"].mean().reset_index()
df = df[df["Year"] >= 2015]
df.head()

,Region,Year,Price ($/MWh)
8,CAISO,2015,57.961429
9,CAISO,2016,38.281818
10,CAISO,2017,33.720000
11,CAISO,2018,32.146000
12,CAISO,2019,23.250000
